### The Two Parts of the Formula:
**TF-IDF** is the product of two distinct metrics:

#### 1. Term Frequency (TF)
This measures how frequently a term occurs in a document.
- **Logic:** The more a word appears in a file, the more likely that file is to be about that word.
- **Calculation:**
$$\text{TF}(t, d) = \frac{\text{Number of times term } t \text{ appears in document } d}{\text{Total number of terms in document } d}$$

#### 2. Inverse Document Frequency (IDF)
This measures how "rare" or "unique" a word is across the entire collection of documents.
- **Logic:** Common words like "the," "is," or "of" appear everywhere but carry no specific meaning. IDF penalizes these common words and rewards rare words (like "Quantum" or "Subramanya").
- **Calculation:**
$$\text{IDF}(t, D) = \log\left(\frac{\text{Total number of documents } D}{\text{Number of documents containing term } t}\right)$$

In [6]:
# this is sklearn based method, python version is written bellow
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# 1. Our "Knowledge Base" Dataset
documents = [
    "Neural Radiance Fields (NeRF) use multilayer perceptrons to represent 3D scenes as continuous volumetric functions.",
    "Gaussian Splatting is a technique for real-time radiance field rendering using 3D Gaussians for faster inference.",
    "3D reconstruction from multi-view images often involves structure-from-motion (SfM) and multi-view stereo (MVS).",
    "Generative AI and diffusion models are transforming how we synthesize novel views and 2D images.",
    "Camera intrinsic parameters and extrinsic matrices are fundamental for reprojecting points from 3D to 2D space."
]

def search_tfidf(query, docs):
    # 2. Initialize the Vectorizer
    # stop_words='english' removes common words like 'is', 'the', 'and'
    vectorizer = TfidfVectorizer(stop_words='english')

    # 3. Fit and Transform the documents into a TF-IDF Matrix
    tfidf_matrix = vectorizer.fit_transform(docs)

    # 4. Transform the search query into the same TF-IDF space
    query_vector = vectorizer.transform([query])

    # 5. Calculate Cosine Similarity between query and all documents
    # This measures the 'angle' between vectors to find the best match
    cosine_similarities = cosine_similarity(query_vector, tfidf_matrix).flatten()

    # 6. Sort and display results
    related_docs_indices = cosine_similarities.argsort()[::-1]
    
    print(f"Search Results for: '{query}'\n" + "-"*30)
    for i in related_docs_indices:
        score = cosine_similarities[i]
        if score > 0:
            print(f"Score: {score:.4f} | Document: {docs[i]}")
        elif i == related_docs_indices[0]:
            print("No relevant documents found.")

# --- Test the search ---
user_query = "rendering 3D scenes with radiance fields"
search_tfidf(user_query, documents)

Search Results for: 'rendering 3D scenes with radiance fields'
------------------------------
Score: 0.4307 | Document: Neural Radiance Fields (NeRF) use multilayer perceptrons to represent 3D scenes as continuous volumetric functions.
Score: 0.2856 | Document: Gaussian Splatting is a technique for real-time radiance field rendering using 3D Gaussians for faster inference.
Score: 0.0505 | Document: Camera intrinsic parameters and extrinsic matrices are fundamental for reprojecting points from 3D to 2D space.
Score: 0.0399 | Document: 3D reconstruction from multi-view images often involves structure-from-motion (SfM) and multi-view stereo (MVS).


In [7]:
# python version
import numpy as np
import math
from collections import Counter

# 1. Define sample documents to search
documents = [
    "3D reconstruction from multi-view images using deep learning frameworks allows for dense point cloud generation.",
    "Multi-view stereo geometry is a foundational concept for creating 3D models from overlapping 2D images.",
    "Deep learning algorithms excel in specific 3D reconstruction tasks, particularly with varied camera extrinsics.",
    "Classic photogrammetry approaches still perform strongly against deep learning when the overlapping viewpoints are sparse.",
    "Generative AI models are opening new pathways to synthesis novel views without traditional 3D point cloud generation."
]

search_query = "How does deep learning help in 3D reconstruction from multi-view images?"

# Combine query and documents
all_text = [search_query] + documents

# 2. Tokenize documents (convert to lowercase and split by space, remove basic punctuation)
import string
def tokenize(text):
    text = text.lower()
    for p in string.punctuation:
        text = text.replace(p, '')
    return text.split()

tokenized_docs = [tokenize(doc) for doc in all_text]

# 3. Create vocabulary and map words to index
vocab = sorted(list(set(word for doc in tokenized_docs for word in doc)))
vocab_index = {word: i for i, word in enumerate(vocab)}
num_docs = len(all_text)
num_terms = len(vocab)

# 4. Calculate Term Frequency (TF) matrix
tf_matrix = np.zeros((num_docs, num_terms))

for i, doc in enumerate(tokenized_docs):
    term_counts = Counter(doc)
    total_terms = len(doc)
    if total_terms == 0:
        continue
    for term, count in term_counts.items():
        # TF = (Number of times term t appears in document d) / (Total terms in d)
        tf_matrix[i, vocab_index[term]] = count / total_terms

# 5. Calculate Inverse Document Frequency (IDF) vector
idf_vector = np.zeros(num_terms)

for term, index in vocab_index.items():
    # Standard IDF calculation: log(Total Documents / Documents containing term)
    # Adding 1 for smoothing similar to sklearn's default
    docs_with_term = sum(1 for doc in tokenized_docs if term in doc)
    idf_vector[index] = math.log((1 + num_docs) / (1 + docs_with_term)) + 1.0

# 6. Calculate final TF-IDF matrix
tfidf_matrix = tf_matrix * idf_vector

# Optional: L2 Normalize the rows (similar to sklearn's default behavior)
norms = np.linalg.norm(tfidf_matrix, axis=1, keepdims=True)
# Avoid division by zero
norms[norms == 0] = 1
tfidf_matrix = tfidf_matrix / norms

# 7. Calculate Cosine Similarity
# Extract the query vector (first row) and document vectors (remaining rows)
query_vector = tfidf_matrix[0]
doc_vectors = tfidf_matrix[1:]

# Cosine similarity: dot product of normalized vectors
# Since we already L2 normalized the rows above, we can just take the dot product
cosine_similarities = np.dot(doc_vectors, query_vector)

# --- Display Results ---
print(f"\nSearch Query: '{search_query}'\n")
print("Ranking Documents:\n")

# Get indices of documents sorted by descending similarity scores
ranked_indices = np.argsort(cosine_similarities)[::-1]

for rank, idx in enumerate(ranked_indices):
    score = cosine_similarities[idx]
    print(f"Rank {rank + 1} (Score: {score:.4f}):")
    print(f"  '{documents[idx]}'\n")


Search Query: 'How does deep learning help in 3D reconstruction from multi-view images?'

Ranking Documents:

Rank 1 (Score: 0.3640):
  '3D reconstruction from multi-view images using deep learning frameworks allows for dense point cloud generation.'

Rank 2 (Score: 0.2481):
  'Deep learning algorithms excel in specific 3D reconstruction tasks, particularly with varied camera extrinsics.'

Rank 3 (Score: 0.1940):
  'Multi-view stereo geometry is a foundational concept for creating 3D models from overlapping 2D images.'

Rank 4 (Score: 0.0761):
  'Classic photogrammetry approaches still perform strongly against deep learning when the overlapping viewpoints are sparse.'

Rank 5 (Score: 0.0268):
  'Generative AI models are opening new pathways to synthesis novel views without traditional 3D point cloud generation.'

